# 08. Trust your experiment

Before believing a result, it pays to check the experiment was run well. The library ships three diagnostics: **SRM** (did the allocation come out as expected?), **A/A** (is the pipeline calibrated?), and **balance** (are covariates balanced? seen in notebook 04).

## SRM: does the allocation match what we intended?

A freshly randomized experiment matches by construction. But if the data pipeline drops units from one arm (logging, bot filtering), the observed proportion departs. `SRMTest` catches it.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.diagnostics import SRMTest

rng = np.random.default_rng(0)
n = 200
df = pd.DataFrame({"x": rng.normal(size=n)})
design = CRD(p=0.5, seed=0)
assignment = design.randomize(df)
print("Freshly randomized, flag:", SRMTest().run(assignment).flagged)

# Simulate losing 40% of the controls (asymmetric data leak)
data = assignment.data_
treated = data[data["treatment"] == 1]
control = data[data["treatment"] == 0].sample(frac=0.6, random_state=0)
corrupted_df = pd.concat([treated, control]).reset_index(drop=True)
corrupted = CRDAssignment(data=corrupted_df, treatment_col="treatment", design=design, seed=0)
res = SRMTest().run(corrupted)
print(f"After losing controls, flag: {res.flagged}  (p={res.p_value:.2e})")

## A/A: is the pipeline calibrated?

An A/A test re-randomizes over data with **no effect** and checks whether the false-positive rate matches `alpha`. A mismatch means something is off in the design or the inference.

In [ ]:
from skxperiments.diagnostics import AATest
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

aa_df = pd.DataFrame({"y": rng.normal(size=200)})   # outcome with no effect
aa = AATest(
    design=CRD(p=0.5, seed=0),
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="y")),
    n_simulations=300,
    seed=0,
)
res_aa = aa.run(aa_df)
print(
    f"False-positive rate: {res_aa.false_positive_rate:.3f} "
    f"(alpha={res_aa.alpha}), flag={res_aa.flagged}"
)

## What we learned

- **SRM** is a cheap, powerful alarm for collection bugs; run it before analyzing.
- **A/A** validates that design + inference are calibrated (false-positive rate near `alpha`).
- **Balance** (notebook 04) completes the trio. In the next notebook, `ExperimentPipeline` runs SRM automatically.

**Next:** `09. Putting it together` plans, runs, and reports an experiment end to end.